In [ ]:
# celda 0

# cuadernillo B para eliminar reacciones que gatillen fugas encontradas en cuadernillo A
# genera archivo JSON con rxnes eliminadas
# demora como 30 min aprox

# montar Drive
from google.colab import drive
drive.mount('/content/drive')

# importar paquetes necesarios
import subprocess
subprocess.run(["pip", "install", "-q", "python-libsbml", "menetools", "clyngor-with-clingo"])

import os
import libsbml
import pandas as pd
from menetools import run_menescope, run_meneacti

def load_model(file_path):
    reader = libsbml.SBMLReader()
    doc = reader.readSBML(file_path)
    if doc.getNumErrors() > 0:
        print(f"Error leyendo {file_path}:")
        doc.printErrors()
        return None
    return doc.getModel()

def buscar_sbml(carpeta):
    for f in os.listdir(carpeta):
        if f.endswith('.sbml'):
            return os.path.join(carpeta, f)
    raise FileNotFoundError(carpeta)

def reaction_to_string(reaction, model):
    """Devuelve la reacción como string estequiométrico."""
    def species_str(spec_ref):
        sp = model.getSpecies(spec_ref.getSpecies())
        coeff = spec_ref.getStoichiometry()
        coeff_str = "" if coeff == 1 else str(coeff) + " "
        return coeff_str + (sp.getId() if sp is not None else spec_ref.getSpecies())

    reactants = " + ".join([species_str(r) for r in reaction.getListOfReactants()])
    products  = " + ".join([species_str(p) for p in reaction.getListOfProducts()])
    return f"{reaction.getId()}: {reactants} -> {products}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# celda 1

# modificar rutas si necesario

MODELOS = '/content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2'
SITIOS = ['Las_Docas', 'Algarrobo', 'Navidad', 'Topocalma', 'Ilque', 'San_Antonio', 'Pargua', 'Los_Chonos']

# buscar_sbml toma el .sbml que haya en la carpeta, sin importar como se llame el archivo

rutas_v1 = {sitio: buscar_sbml(os.path.join(MODELOS, sitio, 'sbml_curado')) for sitio in SITIOS}

CARPETA_TEST = os.path.join(MODELOS, 'COFACTORES', 'test')

# semillas triviales
SEED_WATER  = os.path.join(CARPETA_TEST, 'seed_OnlyWater.sbml')
SEED_AMP    = os.path.join(CARPETA_TEST, 'seed_OnlyAMP.sbml')
SEED_PPI    = os.path.join(CARPETA_TEST, 'seed_OnlyPPI.sbml')
SEED_PROTON = os.path.join(CARPETA_TEST, 'seed_OnlyProton.sbml')

# semilla cofactores
SEED_COF_V1 = os.path.join(MODELOS, 'COFACTORES', 'seed_cofactors.sbml')

print("Rutas v1:", {k: os.path.basename(v) for k, v in rutas_v1.items()})
print("Semillas triviales:")
for nombre, ruta in [('agua', SEED_WATER), ('amp', SEED_AMP), ('ppi', SEED_PPI), ('proton', SEED_PROTON)]:
    print(f"  {nombre}: {ruta}  (existe: {os.path.exists(ruta)})")

Rutas v1: {'Las_Docas': 'ld_metagenome_withgenes.sbml', 'Algarrobo': 'al_metagenome_withgenes.sbml', 'Navidad': 'nav_metagenome_withgenes.sbml', 'Topocalma': 'top_metagenome_withgenes.sbml', 'Ilque': 'ilq_metagenome_withgenes.sbml', 'San_Antonio': 'sant_metagenome_withgenes.sbml', 'Pargua': 'par_metagenome_withgenes.sbml', 'Los_Chonos': 'lc_metagenome_withgenes.sbml'}
Semillas triviales:
  agua: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/test/seed_OnlyWater.sbml  (existe: True)
  amp: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/test/seed_OnlyAMP.sbml  (existe: True)
  ppi: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/test/seed_OnlyPPI.sbml  (existe: True)
  proton: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/COFACTORES/test/seed_OnlyProton.sbml  (existe: True)


In [ ]:
# celda 2

def imprimir_lista_truncada(items, limite=100):
    """Imprime una lista completa, o truncada con aviso si supera 'limite' elementos."""
    if len(items) <= limite:
        for m in items:
            print(" ", m)
    else:
        for m in items[:limite]:
            print(" ", m)
        print(f"   ... ({len(items) - limite} metabolitos más, omitidos por espacio)")


#
def analizar_semilla_grande(nombre_semilla, seed_path, rutas, sitios=SITIOS, limite_impresion=100):
    """
    Analiza una semilla grande (ej. cofactores) usando run_menescope + run_meneacti
    (ASP/clingo). Imprime por sitio: scope, fuga
    (no-cof) y reacciones responsables; devuelve (df_resumen, detalle).
    """
    resumen = []
    detalle = {}

    for site in sitios:
        sbml_path = rutas[site]

        resultado_scope = run_menescope(draft_sbml=sbml_path, seeds_sbml=seed_path)
        scope = sorted(resultado_scope['scope'])
        activadas = run_meneacti(draft_sbml=sbml_path, seeds_sbml=seed_path)
        fuga = sorted([m for m in scope if '__cof__' not in m])

        model = load_model(sbml_path)
        reacciones_fuga = []
        for r in model.getListOfReactions():
            rid = r.getId()
            if rid not in activadas:
                continue
            products_no_cof = [p.getSpecies() for p in r.getListOfProducts() if '__cof__' not in p.getSpecies()]
            # NUEVO: si la reacción es reversible, el lado de los reactivos también puede "producir" en reversa
            reactivos_no_cof = []
            if r.getReversible():
                reactivos_no_cof = [s.getSpecies() for s in r.getListOfReactants() if '__cof__' not in s.getSpecies()]
            if products_no_cof or reactivos_no_cof:
                reacciones_fuga.append(rid)

        print("=" * 70)
        print(f"SITIO: {site}   |   SEMILLA: {nombre_semilla}")
        print("=" * 70)
        print(f"\n--- Metabolitos en el scope ({len(scope)}) ---")
        imprimir_lista_truncada(scope, limite_impresion)
        print(f"\n--- Metabolitos FUGA, no-cof ({len(fuga)}) ---")
        imprimir_lista_truncada(fuga, limite_impresion)
        print(f"\n--- Reacciones responsables de la fuga ({len(reacciones_fuga)}) ---")
        for rid in reacciones_fuga:
            reaction = model.getReaction(rid)
            print("  ", reaction_to_string(reaction, model))

        resumen.append({'sitio': site, 'semilla': nombre_semilla,
                         'n_metabolitos_scope': len(scope), 'n_reacciones_activadas': len(activadas),
                         'n_metabolitos_fuga': len(fuga), 'n_reacciones_fuga': len(reacciones_fuga)})
        detalle[site] = {'scope': scope, 'reacciones_activadas': activadas,
                          'fuga': fuga, 'reacciones_fuga': reacciones_fuga}
        print()

    df_resumen = pd.DataFrame(resumen)
    print("=" * 70)
    print(f"TABLA RESUMEN — semilla '{nombre_semilla}'")
    print("=" * 70)
    print(df_resumen)
    return df_resumen, detalle

In [ ]:
# celda 3

# función de curación manual dirigida

import gc
import json

def curar_por_reacciones(sbml_path_inicial, reacciones_a_eliminar, ronda_label, excepciones=None, verbose=True):
    """
    Elimina, en una sola pasada, exactamente las reacciones de 'reacciones_a_eliminar'
    (elegidas a mano tras ver el diagnóstico de fuga). Regla de seguridad: solo se
    borra si el ID tiene '__cof__', salvo que esté explícitamente en 'excepciones'.
    """
    excepciones = set(excepciones) if excepciones else set()
    model = load_model(sbml_path_inicial)
    eliminadas, omitidas = [], []

    for rid in reacciones_a_eliminar:
        if '__cof__' not in rid and rid not in excepciones:
            omitidas.append(rid)
            continue
        idx = next((i for i in range(model.getListOfReactions().size())
                    if model.getListOfReactions().get(i).getId() == rid), -1)
        if idx >= 0:
            if verbose:
                print("   eliminando:", reaction_to_string(model.getReaction(rid), model))
            model.getListOfReactions().remove(idx)
            eliminadas.append((rid, ronda_label))
        elif verbose:
            print(f"   (no presente en este modelo, se omite: {rid})")

    if omitidas and verbose:
        print(f"   OMITIDAS por no tener __cof__ y no estar en excepciones: {omitidas}")

    return model, eliminadas, omitidas


# reacciones "madre" ya identificadas en el Cuadernillo A (checkpoint)
ruta_checkpoint_madre = os.path.join(MODELOS, 'COFACTORES', 'reacciones_madre_por_sitio.json')
with open(ruta_checkpoint_madre) as f:
    reacciones_madre_por_sitio = json.load(f) # cargadas de cuadernillo A

# añadir excepciones si hay
EXCEPCIONES_SIN_COF = [
    # 'R_ID_SIN_TAG_APROBADA',
]

In [ ]:
# celda 4: ronda 1 de eliminar reacciones "madre". Forma sbml_curado_final
rutas_final = {}
reacciones_eliminadas_por_sitio = {}
omitidas_por_sitio = {}
comparacion = []

for site in SITIOS:
    print("=" * 70)
    print(f"SITIO: {site}")
    print("=" * 70)

    ruta_original = rutas_v1[site]  # solo lectura, viene de sbml_curado
    carpeta_final = os.path.join(MODELOS, site, 'sbml_curado_final')
    os.makedirs(carpeta_final, exist_ok=True)
    ruta_final = os.path.join(carpeta_final, os.path.basename(ruta_original))

    reacciones_a_eliminar = reacciones_madre_por_sitio.get(site, [])
    model_final, eliminadas, omitidas = curar_por_reacciones(
        ruta_original, reacciones_a_eliminar, ronda_label='madre_manual',
        excepciones=EXCEPCIONES_SIN_COF
    )

    libsbml.writeSBMLToFile(model_final.getSBMLDocument(), ruta_final)

    rutas_final[site] = ruta_final
    reacciones_eliminadas_por_sitio[site] = eliminadas
    omitidas_por_sitio[site] = omitidas
    comparacion.append({
        'sitio': site,
        'reacciones_madre_candidatas': len(reacciones_a_eliminar),
        'reacciones_eliminadas': len(eliminadas),
        'reacciones_omitidas_sin_tag': len(omitidas),
    })

    print(f"  Guardado en: {ruta_final}")
    print(f"  (sbml_curado original intacto: {ruta_original})\n")
    del model_final
    gc.collect()

df_comparacion = pd.DataFrame(comparacion)
print(df_comparacion)

with open(os.path.join(MODELOS, 'COFACTORES', 'reacciones_eliminadas_final.json'), 'w') as f:
    json.dump(reacciones_eliminadas_por_sitio, f, indent=2)
print("\nRegistro guardado: reacciones_eliminadas_final.json")

SITIO: Las_Docas
   eliminando: R_1__46__18__46__1__46__2__45__RXN__cof__: M_NADP_e__cof__ + 2.0 M_Reduced__45__ferredoxins_e__cof__ + M_PROTON_e -> M_NADPH_e__cof__ + 2.0 M_Oxidized__45__ferredoxins_e__cof__
   eliminando: R_ACETYL__45__COA__45__ACETYLTRANSFER__45__RXN__cof__: 2.0 M_ACETYL__45__COA_c__cof__ -> M_ACETOACETYL__45__COA_c + M_CO__45__A_c__cof__
   eliminando: R_ADENYL__45__KIN__45__RXN__cof__: M_ATP_c__cof__ + M_AMP_c -> 2.0 M_ADP_c__cof__
   eliminando: R_NADPH__45__DEHYDROGENASE__45__RXN__cof__: M_Acceptor_c__cof__ + M_NADPH_c__cof__ + M_PROTON_c -> M_Donor__45__H2_c__cof__ + M_NADP_c__cof__
   eliminando: R_RXN__45__12444__cof__: M_FMNH2_c__cof__ + M_NADP_c__cof__ -> M_FMN_c__cof__ + M_NADPH_c__cof__ + 2.0 M_PROTON_c
   eliminando: R_RXN0__45__4141__cof__: M_HYDROGEN__45__MOLECULE_c + M_Acceptor_c__cof__ -> M_Donor__45__H2_c__cof__
  Guardado en: /content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2/Las_Docas/sbml_curado_final/ld_metagenome_withgenes.sbml
  (s

In [ ]:
# celda 5

# aqui se ven las reacciones/metabolito fuga post primera curacion

df_cof_final, detalle_cof_final = analizar_semilla_grande(
    'cofactores_final', SEED_COF_V1, rutas_final
)

print("=" * 70)
print("RESUMEN: reacciones eliminadas por sitio (curación manual dirigida)")
print("=" * 70)
for site in SITIOS:
    print(f"  {site}: {len(reacciones_eliminadas_por_sitio[site])} eliminadas | "
          f"{len(omitidas_por_sitio[site])} omitidas (madre sin tag __cof__)")

print("\n" + "=" * 70)
print("VERIFICACIÓN: ¿la semilla de cofactores produce SOLO cofactores?")
print("=" * 70)
for site in SITIOS:
    fuga = detalle_cof_final[site]['fuga']
    if len(fuga) == 0:
        print(f"  {site}: ✅ limpio (fuga=0)")
    else:
        print(f"  {site}: ⚠️ fuga persiste ({len(fuga)} metabolitos): {fuga}")

SITIO: Las_Docas   |   SEMILLA: cofactores_final

--- Metabolitos en el scope (39) ---
  M_ACETALD_c
  M_ACETYL__45__COA_c__cof__
  M_ADENOSINE_c
  M_ADP_c__cof__
  M_AMP_c__cof__
  M_ATP_c__cof__
  M_Acceptor_c__cof__
  M_CO__45__A_c__cof__
  M_Cytochromes__45__C__45__Oxidized_c__cof__
  M_Cytochromes__45__C__45__Oxidized_e__cof__
  M_Cytochromes__45__C__45__Reduced_c__cof__
  M_Cytochromes__45__C__45__Reduced_e__cof__
  M_Donor__45__H2_c__cof__
  M_ETF__45__Oxidized_c__cof__
  M_ETF__45__Reduced_c__cof__
  M_ETOH_c
  M_FADH2_c__cof__
  M_FAD_c__cof__
  M_FMNH2_c__cof__
  M_FMN_c__cof__
  M_GDP_c__cof__
  M_GTP_c__cof__
  M_HYDROGEN__45__MOLECULE_c
  M_NADH__45__P__45__OR__45__NOP_c__cof__
  M_NADH_c__cof__
  M_NADPH_c__cof__
  M_NADP_c__cof__
  M_NAD__45__P__45__OR__45__NOP_c__cof__
  M_NAD_c__cof__
  M_Ox__45__NADPH__45__Hemoprotein__45__Reductases_c__cof__
  M_Ox__45__Thioredoxin_c__cof__
  M_Oxidized__45__Plastocyanins_c__cof__
  M_Oxidized__45__ferredoxins_c__cof__
  M_PROTON_c
 

In [ ]:
# celda 6

# ronda 2: en base a celda 5 se elige a mano las rxnes a borrar:
# las reacciones "derivadas" que siguen produciendo metabolitos no-cof
# parte de sbml_curado_final (ronda 1), genera sbml_curado_final_v2.

REACCIONES_DERIVADAS_A_ELIMINAR = [
    'R_ALCOHOL__45__DEHYDROG__45__RXN__cof__',       # fuga: M_ETOH_c
    'R_ACETALD__45__DEHYDROG__45__RXN__cof__',       # fuga: M_ACETALD_c
    'R_ADENOSINE__45__KINASE__45__RXN__cof__',       # fuga: M_ADENOSINE_c
    'R_HYDROGEN__45__DEHYDROGENASE__45__RXN__cof__', # fuga: M_HYDROGEN__45__MOLECULE_c
    'R_RXN__45__17897__cof__',                       # fuga: protón vía ferredoxinas/NADP
    'R_TRANS__45__RXN0__45__277__cof__',             # fuga: M_PROTON_e
    'R_1__46__12__46__1__46__3__45__RXN__cof__',     # fuga H2, varios sitios
    'R_HYDROG__45__RXN__cof__',                      # fuga H2, solo Topocalma/San_Antonio
]

EXCEPCIONES_SIN_COF_V2 = [
    # 'R_ID_SIN_TAG_APROBADA_V2',
]

rutas_final_v2 = {}
reacciones_eliminadas_v2_por_sitio = {}
omitidas_v2_por_sitio = {}
comparacion_v2 = []

for site in SITIOS:
    print("=" * 70)
    print(f"SITIO: {site}")
    print("=" * 70)

    ruta_original = rutas_final[site]  # partimos de sbml_curado_final (madre ya fuera)
    carpeta_final_v2 = os.path.join(MODELOS, site, 'sbml_curado_final_v2')
    os.makedirs(carpeta_final_v2, exist_ok=True)
    ruta_final_v2 = os.path.join(carpeta_final_v2, os.path.basename(ruta_original))

    model_v2, eliminadas, omitidas = curar_por_reacciones(
        ruta_original, REACCIONES_DERIVADAS_A_ELIMINAR, ronda_label='derivada_manual',
        excepciones=EXCEPCIONES_SIN_COF_V2
    )

    libsbml.writeSBMLToFile(model_v2.getSBMLDocument(), ruta_final_v2)

    rutas_final_v2[site] = ruta_final_v2
    reacciones_eliminadas_v2_por_sitio[site] = eliminadas
    omitidas_v2_por_sitio[site] = omitidas
    comparacion_v2.append({
        'sitio': site,
        'candidatas': len(REACCIONES_DERIVADAS_A_ELIMINAR),
        'eliminadas': len(eliminadas),
        'no_presentes_o_sin_tag': len(REACCIONES_DERIVADAS_A_ELIMINAR) - len(eliminadas),
    })

    print(f"  Guardado en: {ruta_final_v2}")
    print(f"  (sbml_curado_final original intacto: {ruta_original})\n")
    del model_v2
    gc.collect()

df_comparacion_v2 = pd.DataFrame(comparacion_v2)
print(df_comparacion_v2)

with open(os.path.join(MODELOS, 'COFACTORES', 'reacciones_eliminadas_v2.json'), 'w') as f:
    json.dump(reacciones_eliminadas_v2_por_sitio, f, indent=2)
print("\nRegistro guardado: reacciones_eliminadas_v2.json")

SITIO: Las_Docas
   eliminando: R_ALCOHOL__45__DEHYDROG__45__RXN__cof__: M_ETOH_c + M_NAD_c__cof__ -> M_ACETALD_c + M_NADH_c__cof__ + M_PROTON_c
   eliminando: R_ACETALD__45__DEHYDROG__45__RXN__cof__: M_ACETALD_c + M_CO__45__A_c__cof__ + M_NAD_c__cof__ -> M_ACETYL__45__COA_c__cof__ + M_NADH_c__cof__ + M_PROTON_c
   eliminando: R_ADENOSINE__45__KINASE__45__RXN__cof__: M_ADENOSINE_c + M_ATP_c__cof__ -> M_PROTON_c + M_AMP_c__cof__ + M_ADP_c__cof__
   eliminando: R_HYDROGEN__45__DEHYDROGENASE__45__RXN__cof__: M_NAD_c__cof__ + M_HYDROGEN__45__MOLECULE_c -> M_NADH_c__cof__ + M_PROTON_c
   eliminando: R_RXN__45__17897__cof__: 2.0 M_Reduced__45__ferredoxins_c__cof__ + M_NADP_c__cof__ + M_PROTON_c -> 2.0 M_Oxidized__45__ferredoxins_c__cof__ + M_NADPH_c__cof__
   eliminando: R_TRANS__45__RXN0__45__277__cof__: M_NAD_c__cof__ + M_PROTON_c + M_NADPH_c__cof__ -> M_NADP_c__cof__ + M_NADH_c__cof__ + M_PROTON_e
   (no presente en este modelo, se omite: R_1__46__12__46__1__46__3__45__RXN__cof__)
   (no 

In [ ]:
# celda 7

# scope de los modelos sbml_curado_final_v2
# ya no hay fugas

df_cof_v2, detalle_cof_v2 = analizar_semilla_grande(
    'cofactores_final_v2', SEED_COF_V1, rutas_final_v2
)

print("=" * 70)
print("VERIFICACIÓN v2: ¿la semilla de cofactores produce SOLO cofactores?")
print("=" * 70)
for site in SITIOS:
    fuga = detalle_cof_v2[site]['fuga']
    if len(fuga) == 0:
        print(f"  {site}: ✅ limpio (fuga=0)")
    else:
        print(f"  {site}: ⚠️ fuga persiste ({len(fuga)} metabolitos): {fuga}")

SITIO: Las_Docas   |   SEMILLA: cofactores_final_v2

--- Metabolitos en el scope (33) ---
  M_ACETYL__45__COA_c__cof__
  M_ADP_c__cof__
  M_AMP_c__cof__
  M_ATP_c__cof__
  M_Acceptor_c__cof__
  M_CO__45__A_c__cof__
  M_Cytochromes__45__C__45__Oxidized_c__cof__
  M_Cytochromes__45__C__45__Oxidized_e__cof__
  M_Cytochromes__45__C__45__Reduced_c__cof__
  M_Cytochromes__45__C__45__Reduced_e__cof__
  M_Donor__45__H2_c__cof__
  M_ETF__45__Oxidized_c__cof__
  M_ETF__45__Reduced_c__cof__
  M_FADH2_c__cof__
  M_FAD_c__cof__
  M_FMNH2_c__cof__
  M_FMN_c__cof__
  M_GDP_c__cof__
  M_GTP_c__cof__
  M_NADH__45__P__45__OR__45__NOP_c__cof__
  M_NADH_c__cof__
  M_NADPH_c__cof__
  M_NADP_c__cof__
  M_NAD__45__P__45__OR__45__NOP_c__cof__
  M_NAD_c__cof__
  M_Ox__45__NADPH__45__Hemoprotein__45__Reductases_c__cof__
  M_Ox__45__Thioredoxin_c__cof__
  M_Oxidized__45__Plastocyanins_c__cof__
  M_Oxidized__45__ferredoxins_c__cof__
  M_Plastocyanin__45__Reduced_c__cof__
  M_Red__45__NADPH__45__Hemoprotein__45__R

In [ ]:
# celda 8: resumen final reacciones eliminadas
# solo lee los JSON ya guardados en Drive, NO vuelve a correr curar_por_reacciones ni menetools.
# correr celda 0 y 1


import json, re

def decode(raw):
    """Revierte el escapeo SBML (__<codigo_ascii>__ -> caracter) y quita prefijos R_/M_."""
    s = raw
    if s.startswith('R_') or s.startswith('M_'):
        s = s[2:]
    return re.sub(r'__(\d+)__', lambda m: chr(int(m.group(1))), s)

def decode_stoich(stoich_raw):
    """Decodifica una estequiometría completa (reactivos -> productos), término a término."""
    lado_izq, lado_der = stoich_raw.split(' -> ')
    def decode_lado(lado):
        terminos = lado.split(' + ')
        out = []
        for t in terminos:
            partes = t.split(' ', 1)
            if len(partes) == 2 and partes[0].replace('.', '').isdigit():
                coef, especie = partes
                out.append(f"{coef} {decode(especie)}")
            else:
                out.append(decode(t))
        return " + ".join(out)
    return f"{decode_lado(lado_izq)} -> {decode_lado(lado_der)}"

# estequiometría de las 7 reacciones "madre" (Ronda 1), tal como se imprimió en tu celda 4
STOICH_R1 = {
    'R_1__46__18__46__1__46__2__45__RXN__cof__':
        'M_NADP_e__cof__ + 2.0 M_Reduced__45__ferredoxins_e__cof__ + M_PROTON_e -> M_NADPH_e__cof__ + 2.0 M_Oxidized__45__ferredoxins_e__cof__',
    'R_ACETYL__45__COA__45__ACETYLTRANSFER__45__RXN__cof__':
        '2.0 M_ACETYL__45__COA_c__cof__ -> M_ACETOACETYL__45__COA_c + M_CO__45__A_c__cof__',
    'R_ADENYL__45__KIN__45__RXN__cof__':
        'M_ATP_c__cof__ + M_AMP_c -> 2.0 M_ADP_c__cof__',
    'R_NADPH__45__DEHYDROGENASE__45__RXN__cof__':
        'M_Acceptor_c__cof__ + M_NADPH_c__cof__ + M_PROTON_c -> M_Donor__45__H2_c__cof__ + M_NADP_c__cof__',
    'R_RXN__45__12444__cof__':
        'M_FMNH2_c__cof__ + M_NADP_c__cof__ -> M_FMN_c__cof__ + M_NADPH_c__cof__ + 2.0 M_PROTON_c',
    'R_RXN0__45__4141__cof__':
        'M_HYDROGEN__45__MOLECULE_c + M_Acceptor_c__cof__ -> M_Donor__45__H2_c__cof__',
    'R_RXN__45__13750__cof__':
        'M_FADH2_c__cof__ + M_NAD__45__P__45__OR__45__NOP_c__cof__ -> M_FAD_c__cof__ + M_NADH__45__P__45__OR__45__NOP_c__cof__ + 2.0 M_PROTON_c',
}

# estequiometría de las 8 reacciones "derivadas" (Ronda 2), tal como se imprimió en tu celda 6
STOICH_R2 = {
    'R_ALCOHOL__45__DEHYDROG__45__RXN__cof__':
        'M_ETOH_c + M_NAD_c__cof__ -> M_ACETALD_c + M_NADH_c__cof__ + M_PROTON_c',
    'R_ACETALD__45__DEHYDROG__45__RXN__cof__':
        'M_ACETALD_c + M_CO__45__A_c__cof__ + M_NAD_c__cof__ -> M_ACETYL__45__COA_c__cof__ + M_NADH_c__cof__ + M_PROTON_c',
    'R_ADENOSINE__45__KINASE__45__RXN__cof__':
        'M_ADENOSINE_c + M_ATP_c__cof__ -> M_PROTON_c + M_AMP_c__cof__ + M_ADP_c__cof__',
    'R_HYDROGEN__45__DEHYDROGENASE__45__RXN__cof__':
        'M_NAD_c__cof__ + M_HYDROGEN__45__MOLECULE_c -> M_NADH_c__cof__ + M_PROTON_c',
    'R_RXN__45__17897__cof__':
        '2.0 M_Reduced__45__ferredoxins_c__cof__ + M_NADP_c__cof__ + M_PROTON_c -> 2.0 M_Oxidized__45__ferredoxins_c__cof__ + M_NADPH_c__cof__',
    'R_TRANS__45__RXN0__45__277__cof__':
        'M_NAD_c__cof__ + M_PROTON_c + M_NADPH_c__cof__ -> M_NADP_c__cof__ + M_NADH_c__cof__ + M_PROTON_e',
    'R_1__46__12__46__1__46__3__45__RXN__cof__':
        'M_HYDROGEN__45__MOLECULE_c + M_NADP_c__cof__ -> M_PROTON_c + M_NADPH_c__cof__',
    'R_HYDROG__45__RXN__cof__':
        'M_HYDROGEN__45__MOLECULE_c + 2.0 M_Oxidized__45__ferredoxins_c__cof__ -> 2.0 M_Reduced__45__ferredoxins_c__cof__ + 2.0 M_PROTON_c',
}

ruta_json_r1 = os.path.join(MODELOS, 'COFACTORES', 'reacciones_eliminadas_final.json')
ruta_json_r2 = os.path.join(MODELOS, 'COFACTORES', 'reacciones_eliminadas_v2.json')

with open(ruta_json_r1) as f:
    eliminadas_r1 = json.load(f)
with open(ruta_json_r2) as f:
    eliminadas_r2 = json.load(f)

resumen_conteo = []

for site in SITIOS:
    print("=" * 90)
    print(f"SITIO: {site}")
    print("=" * 90)

    r1 = eliminadas_r1.get(site, [])
    r2 = eliminadas_r2.get(site, [])

    print(f"\n-- Ronda 1 (madre): {len(r1)} eliminadas --")
    for rid, _ in r1:
        stq = STOICH_R1.get(rid)
        stq_txt = decode_stoich(stq) if stq else "(estequiometría no cacheada)"
        print(f"   {decode(rid)}")
        print(f"      {stq_txt}")

    print(f"\n-- Ronda 2 (derivada): {len(r2)} eliminadas --")
    for rid, _ in r2:
        stq = STOICH_R2.get(rid)
        stq_txt = decode_stoich(stq) if stq else "(estequiometría no cacheada)"
        print(f"   {decode(rid)}")
        print(f"      {stq_txt}")
    print()

    resumen_conteo.append({
        'sitio': site,
        'eliminadas_ronda1': len(r1),
        'eliminadas_ronda2': len(r2),
        'total': len(r1) + len(r2),
    })

print("=" * 90)
print("RESUMEN")
print("=" * 90)
df_resumen_eliminadas = pd.DataFrame(resumen_conteo)
print(df_resumen_eliminadas)

SITIO: Las_Docas

-- Ronda 1 (madre): 6 eliminadas --
   1.18.1.2-RXN__cof__
      NADP_e__cof__ + 2.0 Reduced-ferredoxins_e__cof__ + PROTON_e -> NADPH_e__cof__ + 2.0 Oxidized-ferredoxins_e__cof__
   ACETYL-COA-ACETYLTRANSFER-RXN__cof__
      2.0 ACETYL-COA_c__cof__ -> ACETOACETYL-COA_c + CO-A_c__cof__
   ADENYL-KIN-RXN__cof__
      ATP_c__cof__ + AMP_c -> 2.0 ADP_c__cof__
   NADPH-DEHYDROGENASE-RXN__cof__
      Acceptor_c__cof__ + NADPH_c__cof__ + PROTON_c -> Donor-H2_c__cof__ + NADP_c__cof__
   RXN-12444__cof__
      FMNH2_c__cof__ + NADP_c__cof__ -> FMN_c__cof__ + NADPH_c__cof__ + 2.0 PROTON_c
   RXN0-4141__cof__
      HYDROGEN-MOLECULE_c + Acceptor_c__cof__ -> Donor-H2_c__cof__

-- Ronda 2 (derivada): 6 eliminadas --
   ALCOHOL-DEHYDROG-RXN__cof__
      ETOH_c + NAD_c__cof__ -> ACETALD_c + NADH_c__cof__ + PROTON_c
   ACETALD-DEHYDROG-RXN__cof__
      ACETALD_c + CO-A_c__cof__ + NAD_c__cof__ -> ACETYL-COA_c__cof__ + NADH_c__cof__ + PROTON_c
   ADENOSINE-KINASE-RXN__cof__
      ADENO